In [0]:

import pyspark.sql.functions as F


def main():
    # dbutils.widgets.text("odate", "2025-09-23")
    # dbutils.widgets.text("catalog", "prd")
    # dbutils.widgets.text("path_orders_raw","/Volumes/prd/stage/raw_data_managed/raw_data/retail/orders_csv/orders")
    # dbutils.widgets.text("table_orders_bronze", "retail_orders")
    

    ODATE = dbutils.widgets.get("odate")
    CATALOG = dbutils.widgets.get("catalog")
    VOL_PATH = dbutils.widgets.get("path_orders_raw")
    TABLE_NAME = dbutils.widgets.get("table_orders_bronze")
    
    ORDERS_BRONZE = f"{CATALOG}.l_bronze.{TABLE_NAME}"
    print(f"Input Path for Retail Orders raw data: {VOL_PATH}")
    print(f"Output Bronze Table for Retail Orders: {ORDERS_BRONZE}")
    print(f"Order Data (Partition): {ODATE}")
    _ = (
        spark
            .read
            .format("csv")
            .option("header", "true")
            .option("inferSchema", "true")
            .load(VOL_PATH)
            .withColumn("dat_ref_carga", F.lit(ODATE))
            .withColumn("ingestion_time", F.current_timestamp())
            .withColumn("file_path", F.col("_metadata.file_path"))
            .write
            .format("delta")
            .partitionBy("dat_ref_carga")
            .mode("overwrite")
            .saveAsTable(ORDERS_BRONZE)
    )
    print(f"Bronze Table: {ORDERS_BRONZE}, partition {ODATE} ingested!")
        
main()

# FIM